# 1.4. Mô hình dữ liệu 5 bảng và kiểm chuẩn toàn cục

## 1. Cấu trúc bảng dữ liệu và cơ chế kiểm chuẩn ngoài bộ nhớ bằng SQLite

Hệ thống tách riêng tầng vận hành nội bộ (`working`) với 5 bảng Parquet chuẩn hóa xuyên suốt Milestone 1 và Milestone 2 (`papers`, `references`, `elements`, `hierarchy`, `bibtex`). Trên quy mô 25.000 bài báo (41.498 phiên bản), công cụ `src/parser/validation.py` xây dựng chỉ mục khóa trên đĩa bằng SQLite để kiểm tra tính duy nhất của khóa chính, toàn vẹn khóa ngoại xuyên bảng, tính duy nhất của nút gốc `document` và tính phi chu trình (DAG) cho từng phiên bản với mức tiêu thụ RAM thấp và ổn định.

In [ ]:
import html
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Rectangle
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

data = json.loads((Path.cwd().parent / "report_summary.json").read_text(encoding="utf-8"))
scope, m1, m2, bench = data["scope"], data["m1"], data["m2"], data["m2_benchmark"]

# Bảng màu học thuật tối giản: 1 tông xanh đậm chủ đạo kết hợp thang xám điềm đạm
BOOK_PALETTE = {
    "navy": "#1B365D",
    "steel": "#3B6E8C",
    "slate": "#64748B",
    "muted": "#94A3B8",
    "light": "#CBD5E1",
    "accent": "#2C5282",
    "grid": "#CBD5E1",
}
MEMBER_STYLES = {
    "DAT": ("#1B365D", "-"),
    "TUAN": ("#3B6E8C", "--"),
    "TAN": ("#4A5568", "-."),
    "TRI": ("#64748B", ":"),
    "TRIET": ("#94A3B8", "-"),
}

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans", "DejaVu Sans", "Arial"],
    "figure.figsize": (9.6, 4.0),
    "figure.dpi": 170,
    "savefig.dpi": 170,
    "savefig.bbox": "tight",
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94A3B8",
    "axes.linewidth": 0.85,
    "axes.titlesize": 10.5,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#1B365D",
    "axes.labelsize": 9.2,
    "axes.labelcolor": "#1E293B",
    "xtick.labelsize": 8.6,
    "ytick.labelsize": 8.6,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "grid.color": "#CBD5E1",
    "grid.linestyle": ":",
    "grid.alpha": 0.50,
})


def fmt_int(x) -> str:
    if x is None or pd.isna(x):
        return "-"
    return f"{int(round(float(x))):,}".replace(",", ".")


def fmt_num(x, digits: int = 2) -> str:
    if x is None or pd.isna(x):
        return "-"
    return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_pct(x, digits: int = 2, scale100: bool = False) -> str:
    if x is None or pd.isna(x):
        return "-"
    val = float(x) * (100.0 if scale100 else 1.0)
    return f"{val:.{digits}f}%".replace(".", ",")


def style_table(
    df: pd.DataFrame,
    caption: str | None = None,
    precision: int = 2,
    int_cols: list[str] | None = None,
    pct_cols: list[str] | None = None,
    pct_scale100: bool = False,
    col_formats: dict[str, object] | None = None,
    **_unused,
):
    # Bảng học thuật kiểu booktabs tối giản: không thanh màu nền, không nút sắp xếp, an toàn HTML 100%.
    out = df.copy()
    int_set = set(int_cols or [])
    pct_set = set(pct_cols or [])
    custom_fmt = dict(col_formats or {})

    num_cols: set[str] = set()
    for c in out.columns:
        if c in int_set or c in pct_set or pd.api.types.is_numeric_dtype(out[c]):
            num_cols.add(c)

    parts = ['<div class="ah-table-scroll"><table class="dataframe ah-clean-table">']
    if caption:
        parts.append(f"<caption>{html.escape(str(caption))}</caption>")

    parts.append("<thead><tr>")
    for col in out.columns:
        cls = ' class="num-col"' if col in num_cols else ""
        parts.append(f"<th{cls}>{html.escape(str(col))}</th>")
    parts.append("</tr></thead><tbody>")

    for _, row in out.iterrows():
        parts.append("<tr>")
        for col in out.columns:
            val = row[col]
            if col in custom_fmt:
                txt = custom_fmt[col](val)
            elif isinstance(val, (list, tuple)):
                txt = ", ".join(str(x) for x in val) if val else "-"
            elif val is None or (not isinstance(val, (str, bytes)) and pd.isna(val)):
                txt = "-"
            elif col in int_set or pd.api.types.is_integer_dtype(out[col]):
                txt = fmt_int(val)
            elif col in pct_set:
                txt = fmt_pct(val, precision, pct_scale100)
            elif pd.api.types.is_float_dtype(out[col]):
                txt = fmt_num(val, precision)
            else:
                txt = str(val)
            cls = ' class="num-col"' if col in num_cols else ""
            esc = html.escape(str(txt))
            if not cls and str(txt).startswith(("http://", "https://", "10.")):
                esc = f'<span class="ah-breakable">{esc}</span>'
            parts.append(f"<td{cls}>{esc}</td>")
        parts.append("</tr>")

    parts.append("</tbody></table></div>")
    return HTML("".join(parts))


def draw_pipeline_diagram(title: str, stages: list[tuple[str, str]]) -> None:
    # Sơ đồ khối quy trình tối giản chuẩn báo cáo khoa học.
    fig, ax = plt.subplots(figsize=(10.0, 2.05))
    ax.set_xlim(0, 10.0)
    ax.set_ylim(0, 2.05)
    ax.axis("off")
    n = len(stages)
    bw = 2.22
    gap = (9.70 - n * bw) / max(n - 1, 1)
    for idx, (st_title, st_desc) in enumerate(stages):
        x0 = 0.15 + idx * (bw + gap)
        ax.add_patch(Rectangle((x0, 0.18), bw, 1.56, facecolor="#FFFFFF", edgecolor="#CBD5E1", linewidth=1.0))
        ax.add_patch(Rectangle((x0, 1.32), bw, 0.42, facecolor="#EDF2F7", edgecolor="#CBD5E1", linewidth=0.8))
        ax.add_patch(Rectangle((x0, 1.32), 0.06, 0.42, facecolor="#1B365D", edgecolor="none"))
        ax.text(x0 + bw / 2 + 0.02, 1.53, st_title, ha="center", va="center", color="#1B365D", fontsize=8.8, fontweight="bold")
        ax.text(x0 + bw / 2, 0.74, st_desc, ha="center", va="center", color="#334155", fontsize=8.0, linespacing=1.35)
        if idx < n - 1:
            ax.add_patch(FancyArrowPatch(
                (x0 + bw + 0.04, 0.96),
                (x0 + bw + gap - 0.04, 0.96),
                arrowstyle="-|>,head_length=5,head_width=3.2",
                color="#64748B",
                lw=1.1,
            ))
    ax.set_title(title, pad=4, fontsize=10.0, fontweight="bold", color="#1B365D")
    plt.tight_layout()
    plt.show()

In [ ]:
for name, rows in m1["schemas"].items():
    df_sc = pd.DataFrame(rows).rename(columns={"name": "Tên cột", "type": "Kiểu dữ liệu PyArrow", "nullable": "Cho phép Null"})
    display(style_table(df_sc, caption=f"Cấu trúc cột chuẩn của bảng {name}."))

row_summary = pd.DataFrame([
    {"Giai đoạn": "M1", "Tên bảng": "papers", "Số bản ghi": scope["papers"], "Khóa chính và ràng buộc kiểm chuẩn": "Khóa (arxiv_id); không trùng lặp chéo giữa 5 thành viên"},
    {"Giai đoạn": "M1", "Tên bảng": "references", "Số bản ghi": m1["arxiv_reference_rows"], "Khóa chính và ràng buộc kiểm chuẩn": "Khóa (arxiv_id, ref_arxiv_id); 0 trích dẫn mồ côi, giữ cả mã trước 2007"},
    {"Giai đoạn": "M2", "Tên bảng": "elements", "Số bản ghi": m2["rows"]["elements"], "Khóa chính và ràng buộc kiểm chuẩn": "Khóa (element_id); khử trùng mã băm SHA-256 xuyên phiên bản (-35,04%)"},
    {"Giai đoạn": "M2", "Tên bảng": "hierarchy", "Số bản ghi": m2["rows"]["hierarchy"], "Khóa chính và ràng buộc kiểm chuẩn": "Khóa (arxiv_id, version, element_id, parent_id); cây DAG 1 gốc, 0 chu trình"},
    {"Giai đoạn": "M2", "Tên bảng": "bibtex", "Số bản ghi": m2["rows"]["bibtex"], "Khóa chính và ràng buộc kiểm chuẩn": "Khóa (arxiv_id, citation_key); hợp nhất trường và đồng bộ khóa trích dẫn (-35,83%)"},
])
display(style_table(
    row_summary,
    caption="Bảng 4.1. Quy mô 5 bảng dữ liệu hợp nhất (tổng cộng 39.196.741 bản ghi ở M1 và M2).",
    int_cols=["Số bản ghi"],
))

df_val = pd.DataFrame([
    ("Toàn vẹn cấu trúc và khóa ngoại 5 bảng (structural_valid)", str(m2["structural_valid"]), "0 lỗi khóa chính, khóa ngoại, gốc document hoặc chu trình"),
    ("Độ phủ phiên bản từ M1 sang M2 (expected so với recorded)", f"{fmt_int(scope['versions'])} / {fmt_int(scope['versions'])}", f"missing_versions = {m2['missing_versions']} (độ phủ 100,00%)"),
    ("Bài báo trùng lặp / trích dẫn mồ côi sau hợp nhất", f"{m1['duplicate_publications_removed']} / {m1['orphan_references_count']}", "Xác minh bảo toàn khóa trên toàn bộ 5 phân vùng"),
    ("Số lỗi cấu trúc phát hiện bởi validate_project", str(len(m2["validation_errors"])), "Đối soát toàn cục bằng chỉ mục SQLite trên đĩa"),
], columns=["Hạng mục kiểm chuẩn", "Kết quả", "Chi tiết xác minh"])
display(style_table(df_val, caption="Bảng 4.2. Kết quả kiểm chuẩn toàn cục validate_project trên 25.000 bài báo."))

fig, ax = plt.subplots(figsize=(9.4, 3.1))
y = np.arange(len(row_summary))
cols_t = [BOOK_PALETTE["navy"] if g == "M1" else BOOK_PALETTE["steel"] for g in row_summary["Giai đoạn"]]
ax.barh(y, row_summary["Số bản ghi"].to_numpy(dtype=float), color=cols_t, height=0.54)
ax.set_yticks(y)
ax.set_yticklabels([f"{r['Tên bảng']} ({r['Giai đoạn']})" for r in row_summary.to_dict("records")])
ax.invert_yaxis()
ax.set_xscale("log")
ax.set_xlim(1e4, 8e7)
ax.grid(axis="x")
ax.set_xlabel("Số bản ghi (thang log10)")
ax.set_title("Hình 4.1. Quy mô 5 bảng dữ liệu chuẩn hóa M1 và M2 (39,20 triệu dòng)")
for i_t, v_t in enumerate(row_summary["Số bản ghi"]):
    ax.text(float(v_t) * 1.18, i_t, fmt_int(v_t), va="center", fontsize=8.2)
plt.tight_layout()
plt.show()

## Tài liệu và mã nguồn đối chiếu

- [Đặc tả chính thức Milestone 1 - Data Crawling](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/).
- [Đặc tả chính thức Milestone 2 - Hierarchical Parsing and Standardization](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html), [API Terms of Use](https://info.arxiv.org/help/api/tou.html) và [OAI-PMH Cutover (`oaipmh.arxiv.org`)](https://info.arxiv.org/help/oa/index.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [`src/crawler/storage.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/storage.py).
- Mã nguồn triển khai: [`src/crawler/merge_shards.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/merge_shards.py).
- Mã nguồn triển khai: [`src/parser/storage.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/storage.py).
- Mã nguồn triển khai: [`src/parser/validation.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/validation.py).
- Mã nguồn triển khai: [`tools/m2_team_consolidator.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/m2_team_consolidator.py).